Q1 a

In [ ]:
import requests
import time

url = "https://data.etagmb.gov.hk/route"
response = requests.get(url)
data = response.json()

ust_routes = ["11", "11M", "11S", "12", "104"]
route_info = {}
regions = data["data"]["routes"]

for region, route_list in regions.items():

    for route_code in route_list:
        if route_code in ust_routes:

            detail_url = f"https://data.etagmb.gov.hk/route/{region}/{route_code}"
            detail_res = requests.get(detail_url).json()

            if detail_res["data"]:
                r_id = detail_res["data"][0]["route_id"]

                if region == "NT" or route_code == "104":
                    route_info[route_code] = r_id
                    print(f"Found: Route {route_code} ({region}) -> ID {r_id}")

print("\nResult:", route_info)

Found: Route 104 (NT) -> ID 2007200
Found: Route 11 (NT) -> ID 2004791
Found: Route 11M (NT) -> ID 2004825
Found: Route 11S (NT) -> ID 2004826
Found: Route 12 (NT) -> ID 2004763

Result: {'104': 2007200, '11': 2004791, '11M': 2004825, '11S': 2004826, '12': 2004763}


b

In [ ]:
import requests

def get_stop_ids():
    target_routes = ["11", "11M", "11S", "12", "104"]
    route_map = {}


    url_route = "https://data.etagmb.gov.hk/route"
    res_route = requests.get(url_route).json()
    regions = res_route["data"]["routes"]

    for region, routes in regions.items():
        for code in routes:
            if code in target_routes:
                url_detail = f"https://data.etagmb.gov.hk/route/{region}/{code}"
                res_detail = requests.get(url_detail).json()
                if res_detail["data"]:

                    route_map[code] = res_detail["data"][0]["route_id"]

    print(f"Current Route IDs: {route_map}")


    print("\n--- Searching for UST Stops ---")
    for code, route_id in route_map.items():
        for direction in [1, 2]:
            url_stop = f"https://data.etagmb.gov.hk/route-stop/{route_id}/{direction}"
            res_stop = requests.get(url_stop).json()

            if "data" not in res_stop:
                continue

            stops = res_stop["data"]["route_stops"]
            for stop in stops:
                name = stop["name_en"]

                if "Hong Kong University of Science and Technology" in name:
                    print(f"Route {code} (Dir {direction}): {name} -> stop_id: {stop['stop_id']}")

get_stop_ids()

Current Route IDs: {'12': 2004763, '104': 2007200, '11': 2004791, '11M': 2004825, '11S': 2004826}

--- Searching for UST Stops ---
Route 104 (Dir 1): Hong Kong University of Science and Technology (South Station) -> stop_id: 20015226
Route 104 (Dir 1): Hong Kong University of Science and Technology (South Station) -> stop_id: 20015226
Route 11 (Dir 1): Hong Kong University of Science and Technology (South Station) -> stop_id: 20013010
Route 11 (Dir 2): Hong Kong University of Science and Technology (North Station) -> stop_id: 20012472
Route 11M (Dir 1): Hong Kong University of Science and Technology (North Station) -> stop_id: 20012474
Route 11M (Dir 2): Hong Kong University of Science and Technology (North Station) -> stop_id: 20012474
Route 11S (Dir 1): Hong Kong University of Science and Technology (South Station) -> stop_id: 20013011
Route 11S (Dir 2): Hong Kong University of Science and Technology (North Station) -> stop_id: 20012474


c

In [ ]:
import requests
import pandas as pd
from datetime import datetime

def get_timetable(direction):
    if direction == "N":
        target_keywords = ["SCIENCE", "TECHNOLOGY", "UNIVERSITY"]
        exclude_keyword = "SOUTH"
    elif direction == "S":
        target_keywords = ["SCIENCE", "TECHNOLOGY", "UNIVERSITY"]
        exclude_keyword = "NORTH"
    else:
        print("No such direction")
        return None

    target_routes = ["11", "11M", "11S", "12", "104"]
    default_dests = {
        "11": "Choi Hung", "11M": "Hang Hau", "11S": "Choi Hung",
        "12": "Sai Kung", "104": "Ngau Tau Kok"
    }

    route_map = {}
    try:
        url_route = "https://data.etagmb.gov.hk/route"
        res_route = requests.get(url_route).json()
        if "data" in res_route:
            for region, routes in res_route["data"]["routes"].items():
                for code in routes:
                    if code in target_routes:
                        detail_url = f"https://data.etagmb.gov.hk/route/{region}/{code}"
                        detail_res = requests.get(detail_url).json()
                        if detail_res["data"]:
                            route_map[code] = detail_res["data"][0]["route_id"]
    except Exception:
        return None

    timetable_list = []

    for code, route_id in route_map.items():
        for seq in [1, 2]:
            try:
                url_stops = f"https://data.etagmb.gov.hk/route-stop/{route_id}/{seq}"
                res_stops = requests.get(url_stops).json()

                if "data" not in res_stops:
                    continue

                found_stop_id = None

                for stop in res_stops["data"]["route_stops"]:
                    name_upper = stop["name_en"].upper()
                    if any(k in name_upper for k in target_keywords):
                        if exclude_keyword in name_upper:
                            continue
                        found_stop_id = stop["stop_id"]
                        break

                if found_stop_id:
                    url_eta = f"https://data.etagmb.gov.hk/eta/route-stop/{route_id}/{found_stop_id}"
                    res_eta = requests.get(url_eta).json()

                    if res_eta.get("data") and isinstance(res_eta["data"], list):
                        for segment in res_eta["data"]:
                            if "eta" in segment and segment["eta"]:
                                for entry in segment["eta"]:
                                    dest = entry.get("dest_en")
                                    if not dest:
                                        dest = default_dests.get(code, "Unknown")

                                    if "SCIENCE" in dest.upper() or "UNIVERSITY" in dest.upper():
                                        continue

                                    timestamp_str = entry.get("timestamp")
                                    if timestamp_str:
                                        dt = datetime.fromisoformat(timestamp_str)
                                        time_display = dt.strftime("%H:%M")

                                        timetable_list.append({
                                            "route_code": code,
                                            "dest_en": dest,
                                            "ETA": time_display,
                                            "raw_time": dt
                                        })
            except Exception:
                continue

    if not timetable_list:
        return pd.DataFrame(columns=["route_code", "dest_en", "ETA"])

    df = pd.DataFrame(timetable_list)
    df = df.sort_values(by="raw_time")

    final_df = df[["route_code", "dest_en", "ETA"]].reset_index(drop=True)

    return final_df

print("=== UST North Gate Timetable ===")
print(get_timetable("N"))

print("\n=== UST South Gate Timetable ===")
print(get_timetable("S"))

=== UST North Gate Timetable ===
   route_code    dest_en    ETA
0         11M   Hang Hau  19:37
1         11M   Hang Hau  19:37
2         11M   Hang Hau  19:42
3         11M   Hang Hau  19:42
4          11  Choi Hung  19:43
5         11M   Hang Hau  19:45
6         11M   Hang Hau  19:45
7         11M   Hang Hau  19:45
8         11M   Hang Hau  19:45
9          11  Choi Hung  19:49
10        11M   Hang Hau  19:50
11        11M   Hang Hau  19:50
12        11M   Hang Hau  19:53
13        11M   Hang Hau  19:53
14         11  Choi Hung  19:53

=== UST South Gate Timetable ===
  route_code       dest_en    ETA
0        104  Ngau Tau Kok  19:40
1         11     Choi Hung  19:41
2        104  Ngau Tau Kok  19:45
3         11     Choi Hung  19:47
4         11     Choi Hung  19:52
5        104  Ngau Tau Kok  20:10
6        104  Ngau Tau Kok  20:30
7        104  Ngau Tau Kok  20:35


Q2 a

In [ ]:
import requests
import pandas as pd
import time

def scrap_weather_data():
    base_url = "https://www.hko.gov.hk/cis/dailyExtract/dailyExtract_{}.xml"
    data_list = []

    for year in range(2000, 2026):
        try:
            url = base_url.format(year)
            response = requests.get(url)

            if response.status_code == 200:
                raw_data = response.json()
                months_list = raw_data["stn"]["data"]

                for index, month_data in enumerate(months_list):
                    month = index + 1

                    if year == 2000 and month < 11:
                        continue
                    if year == 2025 and month > 10:
                        continue

                    daily_records = month_data.get("dayData", [])

                    for record in daily_records:
                        if not record:
                            continue

                        day_val = record[0]

                        if not str(day_val).isdigit():
                            continue

                        rainfall_val = record[8]
                        if rainfall_val == "Trace":
                            rainfall_val = 0.02

                        entry = {
                            "Date": f"{year}-{month:02d}-{int(day_val):02d}",
                            "Mean Pressure": record[1],
                            "Absolute Daily Max Air Temp.": record[2],
                            "Mean Air Temp.": record[3],
                            "Absolute Daily Min Air Temp.": record[4],
                            "Mean Dew Point": record[5],
                            "Mean Relative Humidity": record[6],
                            "Mean Amount of Cloud": record[7],
                            "Total Rainfall": rainfall_val,
                            "Total Bright Sunshine": record[9]
                        }
                        data_list.append(entry)

            time.sleep(0.1)

        except Exception as e:
            print(f"Error: {e}")

    df = pd.DataFrame(data_list)

    numeric_columns = [
        "Mean Pressure",
        "Absolute Daily Max Air Temp.",
        "Mean Air Temp.",
        "Absolute Daily Min Air Temp.",
        "Mean Dew Point",
        "Mean Relative Humidity",
        "Mean Amount of Cloud",
        "Total Rainfall",
        "Total Bright Sunshine"
    ]

    for col in numeric_columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    df.to_csv("HKO_Weather_2000_2025.csv", index=False)
    return df

weather_df = scrap_weather_data()
print(weather_df.head())
print(len(weather_df))

         Date  Mean Pressure  Absolute Daily Max Air Temp.  Mean Air Temp.  \
0  2000-11-01         1013.3                          23.3            20.7   
1  2000-11-02         1017.3                          23.7            19.9   
2  2000-11-03         1016.8                          23.9            21.5   
3  2000-11-04         1015.6                          23.3            22.2   
4  2000-11-05         1011.7                          25.8            23.5   

   Absolute Daily Min Air Temp.  Mean Dew Point  Mean Relative Humidity  \
0                          18.6             9.0                      47   
1                          16.9             9.1                      50   
2                          18.6            12.6                      57   
3                          20.9            13.4                      58   
4                          21.4            17.0                      67   

   Mean Amount of Cloud  Total Rainfall  Total Bright Sunshine  
0              

b

In [ ]:
import requests
import csv
import os

save_dir = '.'

warnings_config = [
    {
        "name": "Very Hot Weather Warning",
        "url": "https://www.hko.gov.hk/dps/wxinfo/climat/warndb/hot.dat",
        "output_file": "Very_Hot_Weather_Warning.csv",
        "has_colour": False
    },
    {
        "name": "Cold Weather Warning",
        "url": "https://www.hko.gov.hk/dps/wxinfo/climat/warndb/cold.dat",
        "output_file": "Cold_Weather_Warning.csv",
        "has_colour": False
    },
    {
        "name": "Fire Danger Warning",
        "url": "https://www.hko.gov.hk/dps/wxinfo/climat/warndb/fire.dat",
        "output_file": "Fire_Danger_Warning.csv",
        "has_colour": True,
        "color_map": {'Y': 'Yellow', 'R': 'Red', '1': 'Yellow', '2': 'Red'}
    },
    {
        "name": "Rainstorm Warning",
        "url": "https://www.hko.gov.hk/dps/wxinfo/climat/warndb/rstorm.dat",
        "output_file": "Rainstorm_Warning.csv",
        "has_colour": True,
        "color_map": {'A': 'Amber', 'R': 'Red', 'B': 'Black', '1': 'Amber', '2': 'Red', '3': 'Black'}
    }
]

print("Starting data retrieval...")

for config in warnings_config:
    print(f"Processing {config['name']}...")

    try:
        response = requests.get(config['url'])
        response.raise_for_status()
        response.encoding = 'utf-8'
        lines = response.text.splitlines()

        file_path = os.path.join(save_dir, config['output_file'])

        with open(file_path, 'w', newline='', encoding='utf-8') as f:
            writer = csv.writer(f)

            header = ['start_datetime', 'end_datetime']
            if config['has_colour']:
                header.append('colour')
            writer.writerow(header)

            count = 0

            for line in lines:
                parts = line.split()

                if not parts:
                    continue

                time_parts = []
                color_val = "Unknown"

                if parts[0].isalpha():
                    if config['has_colour']:
                        raw_code = parts[0]
                        color_val = config['color_map'].get(raw_code, raw_code)

                    if len(parts) >= 11:
                        time_parts = parts[1:11]
                    else:
                        continue

                elif parts[0].isdigit():
                    if len(parts) >= 10:
                        time_parts = parts[0:10]
                    else:
                        continue

                    if config['has_colour'] and len(parts) > 10:
                        raw_code = parts[10]
                        color_val = config['color_map'].get(raw_code, raw_code)

                else:
                    continue

                if not all(p.isdigit() for p in time_parts):
                    continue

                year = int(time_parts[0])
                month = int(time_parts[1])

                if (year == 2000 and month >= 11) or (2000 < year < 2025) or (year == 2025 and month <= 10):

                    s_year, s_mon, s_day, s_hr, s_min = time_parts[0:5]
                    e_year, e_mon, e_day, e_hr, e_min = time_parts[5:10]

                    start_dt = f"{s_year}-{s_mon.zfill(2)}-{s_day.zfill(2)} {s_hr.zfill(2)}:{s_min.zfill(2)}:00"
                    end_dt = f"{e_year}-{e_mon.zfill(2)}-{e_day.zfill(2)} {e_hr.zfill(2)}:{e_min.zfill(2)}:00"

                    row = [start_dt, end_dt]

                    if config['has_colour']:
                        row.append(color_val)

                    writer.writerow(row)
                    count += 1

            print(f"  Saved {count} records to {config['output_file']}")

    except Exception as e:
        print(f"  Error fetching {config['url']}: {e}")

print("Done.")

Starting data retrieval...
Processing Very Hot Weather Warning...
  Saved 467 records to Very_Hot_Weather_Warning.csv
Processing Cold Weather Warning...
  Saved 160 records to Cold_Weather_Warning.csv
Processing Fire Danger Warning...
  Saved 1148 records to Fire_Danger_Warning.csv
Processing Rainstorm Warning...
  Saved 797 records to Rainstorm_Warning.csv
Done.


c

In [31]:
import pandas as pd
import numpy as np

main_weather_file = 'HKO_Weather_2000_2025.csv'
warning_files = {
    'VHW': 'Very_Hot_Weather_Warning.csv',
    'CWW': 'Cold_Weather_Warning.csv',
    'FDW': 'Fire_Danger_Warning.csv',
    'RW': 'Rainstorm_Warning.csv'
}

df_weather = pd.read_csv(main_weather_file)
df_weather['Date'] = pd.to_datetime(df_weather['Date'])

df_weather['Very Hot Weather Warning'] = False
df_weather['Cold Weather Warning'] = False
df_weather['Fire Danger Warning'] = 0
df_weather['Rainstorm Warning'] = 0

fire_map = {'Yellow': 1, 'Red': 2, 'Y': 1, 'R': 2}
rain_map = {'Amber': 1, 'Red': 2, 'Black': 3, 'A': 1, 'R': 2, 'B': 3}

def apply_warning_to_weather(df_main, filepath, target_col, mode, mapping=None):
    df_warn = pd.read_csv(filepath)

    df_warn['start_datetime'] = pd.to_datetime(df_warn['start_datetime'], errors='coerce')
    df_warn['end_datetime'] = pd.to_datetime(df_warn['end_datetime'], errors='coerce')
    df_warn.dropna(subset=['start_datetime', 'end_datetime'], inplace=True)

    for _, row in df_warn.iterrows():

        start_date = row['start_datetime'].normalize()
        end_date = row['end_datetime'].normalize()

        mask = (df_main['Date'] >= start_date) & (df_main['Date'] <= end_date)

        if not mask.any():
            continue

        if mode == 'bool':
            df_main.loc[mask, target_col] = True

        elif mode == 'level':
            color_val = str(row['colour']).strip()
            level = mapping.get(color_val, 0)

            current_levels = df_main.loc[mask, target_col]
            df_main.loc[mask, target_col] = np.maximum(current_levels, level)

print("Starting data merging...")

apply_warning_to_weather(df_weather, warning_files['VHW'], 'Very Hot Weather Warning', 'bool')
apply_warning_to_weather(df_weather, warning_files['CWW'], 'Cold Weather Warning', 'bool')
apply_warning_to_weather(df_weather, warning_files['FDW'], 'Fire Danger Warning', 'level', fire_map)
apply_warning_to_weather(df_weather, warning_files['RW'], 'Rainstorm Warning', 'level', rain_map)

output_filename = 'HKO_Weather_Final_Combined.csv'
df_weather.to_csv(output_filename, index=False)

print("\nData merging successfully completed.")
print(f"Final file saved as: {output_filename}")

Starting data merging...

Data merging successfully completed.
Final file saved as: HKO_Weather_Final_Combined.csv


d

In [29]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

FILE_PATH = 'HKO_Weather_Final_Combined.csv'
OUTPUT_DIR = 'q2d_figures'

FEATURES = [
    'Mean Pressure',
    'Absolute Daily Max Air Temp.',
    'Mean Air Temp.',
    'Absolute Daily Min Air Temp.',
    'Mean Dew Point',
    'Mean Relative Humidity',
    'Mean Amount of Cloud',
    'Total Rainfall',
    'Total Bright Sunshine'
]

WARNING_ANALYSIS = {
    'Very Hot Weather Warning': {
        'column': 'Very Hot Weather Warning',
        'type': 'bool',
        'levels': [
            {'label': 'No Warning', 'value': False, 'mask_func': lambda df: df['Very Hot Weather Warning'] == False},
            {'label': 'With Warning', 'value': True, 'mask_func': lambda df: df['Very Hot Weather Warning'] == True}
        ],
        'output_prefix': 'VHW_Distribution'
    },
    'Cold Weather Warning': {
        'column': 'Cold Weather Warning',
        'type': 'bool',
        'levels': [
            {'label': 'No Warning', 'value': False, 'mask_func': lambda df: df['Cold Weather Warning'] == False},
            {'label': 'With Warning', 'value': True, 'mask_func': lambda df: df['Cold Weather Warning'] == True}
        ],
        'output_prefix': 'CWW_Distribution'
    },
    'Fire Danger Warning': {
        'column': 'Fire Danger Warning',
        'type': 'level',
        'levels': [
            {'label': 'No Warning', 'value': 0, 'mask_func': lambda df: df['Fire Danger Warning'] == 0},
            {'label': 'Yellow', 'value': 1, 'mask_func': lambda df: df['Fire Danger Warning'] == 1},
            {'label': 'Red', 'value': 2, 'mask_func': lambda df: df['Fire Danger Warning'] == 2}
        ],
        'output_prefix': 'FDW_Distribution'
    },
    'Rainstorm Warning': {
        'column': 'Rainstorm Warning',
        'type': 'level',
        'levels': [
            {'label': 'No Warning', 'value': 0, 'mask_func': lambda df: df['Rainstorm Warning'] == 0},
            {'label': 'Amber', 'value': 1, 'mask_func': lambda df: df['Rainstorm Warning'] == 1},
            {'label': 'Red/Black', 'value': 2, 'mask_func': lambda df: df['Rainstorm Warning'] >= 2}
        ],
        'output_prefix': 'RW_Distribution'
    }
}

def create_histograms(df, analysis_config, warning_name):
    if not os.path.exists(OUTPUT_DIR):
        os.makedirs(OUTPUT_DIR)

    fig, axes = plt.subplots(3, 3, figsize=(15, 15))
    axes = axes.flatten()

    for i, feature in enumerate(FEATURES):
        ax = axes[i]
        all_data = []

        for level in analysis_config['levels']:
             data = df.loc[level['mask_func'](df), feature].dropna()
             if feature == 'Total Rainfall':
                  data = data[data > 0]
             if not data.empty:
                  all_data.append(data)

        if all_data:
            min_val = min(d.min() for d in all_data)
            max_val = max(d.max() for d in all_data)

            for level in analysis_config['levels']:
                data = df.loc[level['mask_func'](df), feature].dropna()

                is_log_plot = (feature == 'Total Rainfall')

                if is_log_plot:
                     data_to_plot = data[data > 0]
                     if not data_to_plot.empty:
                         log_min = np.log10(data_to_plot.min())
                         log_max = np.log10(data_to_plot.max())
                         log_bins = np.logspace(log_min, log_max, 30)

                         ax.hist(data_to_plot, bins=log_bins, density=True, label=level['label'], alpha=0.6, histtype='stepfilled')
                         ax.set_xscale('log')
                         ax.set_yscale('log')
                         ax.set_ylabel('Log Frequency')
                         ax.set_xlabel(f"{feature} (Log Scale)")
                else:
                    if not data.empty:
                        bins = np.linspace(min_val, max_val, 30)
                        ax.hist(data, bins=bins, density=True, label=level['label'], alpha=0.6, histtype='stepfilled')
                        ax.set_ylabel('Frequency')
                        ax.set_xlabel(feature)

        ax.set_title(feature, fontsize=10)
        ax.legend(fontsize=8, loc='best')
        ax.grid(axis='y', alpha=0.5)

    fig.suptitle(f'Distributions of Meteorological Parameters under {warning_name}', fontsize=16)
    plt.tight_layout(rect=[0, 0.03, 1, 0.98])

    output_filename = os.path.join(OUTPUT_DIR, f'{analysis_config["output_prefix"]}_Analysis.png')
    plt.savefig(output_filename)
    print(f"Figure saved: {output_filename}")
    plt.close(fig)

def main():
    try:
        print(f"Loading data from: {FILE_PATH}...")
        df = pd.read_csv(FILE_PATH)

        required_cols = [c['column'] for c in WARNING_ANALYSIS.values()]
        if not all(col in df.columns for col in required_cols):
            print("ERROR: Missing required warning columns. Please check previous steps.")
            return

        for feature in FEATURES:
             df[feature] = pd.to_numeric(df[feature], errors='coerce')

        for analysis_name, config in WARNING_ANALYSIS.items():
            create_histograms(df.copy(), config, analysis_name)

        print("\nAll 4 analysis figures have been generated successfully.")

    except FileNotFoundError:
        print(f"\nERROR: The file '{FILE_PATH}' was not found.")
    except Exception as e:
        print(f"\nAn unexpected error occurred: {e}")

if __name__ == '__main__':
    main()

Loading data from: HKO_Weather_Final_Combined.csv...
Figure saved: q2d_figures/VHW_Distribution_Analysis.png
Figure saved: q2d_figures/CWW_Distribution_Analysis.png
Figure saved: q2d_figures/FDW_Distribution_Analysis.png
Figure saved: q2d_figures/RW_Distribution_Analysis.png

All 4 analysis figures have been generated successfully.
